In [4]:
### Merge and export feature_table and taxonomy 

# Script prepared for last version of dada2  - columns name sequences 

import pandas as pd
from pathlib import Path

In [5]:
# 1. Load files 
base_path = Path("/home/tatiana/PROYECTOS/repository") # change by your path

ft = pd.read_csv(base_path / "feature_table.csv")
tx = pd.read_csv(base_path / "taxonomy.csv")

# Automatically detect the name of the first column (sample names)
sample_col_name = ft.columns[0]
print(f"Sample column detected in feature table: '{sample_col_name}'")

# === 2. Transpose the abundance table ===
samples = ft[sample_col_name].tolist()

# Copy the DataFrame excluding the sample column and set sample names as index
ft2 = ft.drop(columns=[sample_col_name]).copy()
ft2.index = samples

# Transpose so features (sequences) become rows and samples become columns
ft_t = ft2.T
ft_t.index.name = "Feature"
ft_t = ft_t.reset_index()

# === 3. Prepare the taxonomy table ===
# Ensure the sequence column in taxonomy is named 'Feature'
if "Sequence" in tx.columns:
    tx2 = tx.rename(columns={"Sequence": "Feature"}).copy()
elif "feature-id" in tx.columns:
    tx2 = tx.rename(columns={"feature-id": "Feature"}).copy()
else:
    # If it has another name, take the first column for safety
    tx2 = tx.copy()
    tx2.rename(columns={tx2.columns[0]: "Feature"}, inplace=True)

# Remove accidental spaces in the sequences of both dataframes for a perfect match
ft_t["Feature"] = ft_t["Feature"].astype(str).str.strip()
tx2["Feature"] = tx2["Feature"].astype(str).str.strip()

# Pre-check for matches
seq_abund = set(ft_t["Feature"])
seq_tax = set(tx2["Feature"])
print(f"\nExact ASV matches: {len(seq_abund & seq_tax)} out of {len(seq_abund)}")

# 4. Merge abundances and taxonomy 
merged = ft_t.merge(
    tx2,
    on="Feature",
    how="left",
    validate="one_to_one"
)

# 5. Reorder columns 
tax_cols = [
    "Feature", "Kingdom", "Phylum", "Class",
    "Order", "Family", "Genus", "Species"
]

# Filter only the taxonomic and sample columns that actually exist in the merged table
tax_cols_present = [c for c in tax_cols if c in merged.columns]
sample_cols_present = [c for c in samples if c in merged.columns]

merged = merged[tax_cols_present + sample_cols_present]

# 6. Check the merge
print("\nFinal dimensions:", merged.shape)
print("ASVs without taxonomy (Kingdom NaN):", merged["Kingdom"].isna().sum())

print("\nFirst rows:")
display(merged.head())

# Export the file
output_file = base_path / "merged_feature_taxonomy_fungil.csv"
merged.to_csv(output_file, index=False)

print(f"\nFile successfully saved to: {output_file}")

Sample column detected in feature table: 'Unnamed: 0'

Exact ASV matches: 1023 out of 1023

Final dimensions: (1023, 17)
ASVs without taxonomy (Kingdom NaN): 0

First rows:


,Feature,Kingdom,Phylum,Class,Order,Family,Genus,Species,its-LC1,its-LC2,its-LC3,its-LL1,its-LL2,its-LL3,its-LRS1,its-LRS2,its-LRS3
0,TCGAGTTTACAACTCCCAAAACCCCTGTGAACATACCTATCGTTGC...,k__Fungi,p__Ascomycota,c__Sordariomycetes,o__Hypocreales,f__Nectriaceae,g__Paracremonium,NaN,0,325,147,18,35607,11816,0,0,350477
1,CCGAGTTTACAACTCCCAAACCCCTGTGAACATACCTATCGTTGCT...,k__Fungi,p__Ascomycota,c__Sordariomycetes,o__Hypocreales,f__Nectriaceae,g__Cosmospora,NaN,0,87,439,0,26516,5610,0,889,283748
2,ACAAATCCTGCTGCAGTACCGCAAGGTGCTGTAGTATATTAACTCG...,k__Fungi,p__Rozellomycota,c__Rozellomycota_cls_Incertae_sedis,o__Rozellomycota_ord_Incertae_sedis,f__Rozellomycota_fam_Incertae_sedis,g__Rozellomycota_gen_Incertae_sedis,NaN,62678,152077,39725,3061,0,2755,0,0,1205
3,AGAGAATATTGGACTTTGGTCCATTTATCTACCCATCTACACCTGT...,k__Fungi,p__Basidiomycota,c__Tremellomycetes,o__Tremellales,f__Cryptococcaceae,g__Cryptococcus,s__neoformans,0,584,8,0,309,234,0,0,0
4,ACAAATCTATAGGCGGCCTAGCCGTCGTTAACCAATGTGAATTGGG...,k__Fungi,NaN,NaN,NaN,NaN,NaN,NaN,30589,53145,8192,2273,0,4049,0,0,212



File successfully saved to: /home/tatiana/PROYECTOS/repository/merged_feature_taxonomy_fungil.csv
